In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Loan_Default_Prediction\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Loan_Default_Prediction'

In [5]:
import mlflow
mlflow.__version__

'2.10.2'

In [6]:
import dagshub

In [7]:
dagshub.__version__

'0.7.0'

In [8]:
#https://dagshub.com/ajaychaudhary8104/End_to_End_ML_project_for_Loan_Default_Prediction.mlflow
#ajaychaudhary8104

In [ ]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/ajaychaudhary8104/End_to_End_ML_project_for_Loan_Default_Prediction.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="ajaychaudhary8104"
os.environ["MLFLOW_TRACKING_PASSWORD"]="password"

In [10]:
import dagshub
dagshub.init(repo_owner='ajaychaudhary8104', repo_name='End_to_End_ML_project_for_Loan_Default_Prediction', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\Hp\anaconda3\envs\loan\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=35d3f390-d1a3-42b2-bd5d-1e480777299f&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=7dc22fd0ca05f28b578b36473c28ea309b4f94e2261dbc58f041427b36ff83f8




Accessing as ajaychaudhary8104

Initialized MLflow to track repo "ajaychaudhary8104/End_to_End_ML_project_for_Loan_Default_Prediction"

Repository ajaychaudhary8104/End_to_End_ML_project_for_Loan_Default_Prediction initialized!

In [11]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [12]:
from src.loan_default_prediction.constants import *
from src.loan_default_prediction.utils.common import read_yaml, create_directories, save_json

In [13]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params= dict(params.model_params),
            metric_file_name = config.metric_file_name,
            target_column = config.target_column,
            mlflow_uri= config.mlflow_uri
        )

        return model_evaluation_config

In [14]:
import os
import pandas as pd
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score, classification_report, roc_auc_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [16]:
import pickle as pkl

In [19]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    
    def eval_metrics(self,actual, pred):
        precision = precision_score(actual, pred, average='weighted')
        recall = recall_score(actual, pred, average='weighted')
        f1 = f1_score(actual, pred, average='weighted')
        accuracy = accuracy_score(actual, pred)
        roc_auc = roc_auc_score(actual, pred)
        return (precision, recall, f1, accuracy, roc_auc)
    


    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = pkl.load(open(self.config.model_path, "rb"))

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]


        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme


        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (precision, recall, f1, accuracy, roc_auc) = self.eval_metrics(test_y, predicted_qualities)
            
            # Saving metrics as local
            scores = {"precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy, "roc_auc": roc_auc}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1", f1)
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("roc_auc", roc_auc)


            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="XGBClassifier")
            else:
                mlflow.sklearn.log_model(model, "model")

In [20]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-05-03 16:36:20,091: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-03 16:36:20,096: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-03 16:36:20,098: INFO: common: created directory at: artifacts]
[2026-05-03 16:36:20,100: INFO: common: created directory at: artifacts/model_evaluation]


[2026-05-03 16:36:22,636: INFO: common: json file saved at: artifacts\model_evaluation\model_evaluation_metrics.json]


Successfully registered model 'XGBClassifier'.
2026/05/03 16:36:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBClassifier, version 1
Created version '1' of model 'XGBClassifier'.
